# QuantHive Multi-Agent Training Notebook

Rerunnable notebook for validating the PettingZoo environment, running the rule-based multi-agent trainer, previewing governance-aware prompts, and optionally launching GRPO training on a GPU runtime.


## 1. Bootstrap The Repo

Clone from GitHub when the notebook is running outside the repo. If the current working directory already looks like the repository, reuse it.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ARKAISW/multi-agent-trading-env.git"

def looks_like_repo(path: Path) -> bool:
    return (
        (path / "openenv.yaml").exists()
        and (path / "env").exists()
        and (path / "training").exists()
    )

start_dir = Path.cwd()
if looks_like_repo(start_dir):
    REPO_DIR = start_dir
else:
    clone_parent = Path("/content") if Path("/content").exists() else start_dir
    REPO_DIR = clone_parent / "multi-agent-trading-env"
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Working directory: {Path.cwd()}")
print(f"Repo commit: {commit}")
print(f"Python: {sys.version.split()[0]}")


## 2. Install Notebook Dependencies

Install the lightweight stack needed for environment validation, rule-based training, plots, and prompt preview. The heavier GRPO packages stay in the optional GPU section.


In [ ]:
import importlib.metadata as importlib_metadata

BASE_PACKAGES = [
    "openenv",
    "pyyaml",
    "pettingzoo>=1.24,<1.26",
    "gymnasium",
    "numpy",
    "pandas",
    "matplotlib",
    "scipy",
    "torch",
    "yfinance",
    "ccxt",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *BASE_PACKAGES])
print("Installed base notebook dependencies.")
print(f"PettingZoo version: {importlib_metadata.version('pettingzoo')}")


## 2.5. Apply Hosted Runtime Compatibility Patch

When this notebook clones an older repo snapshot, patch the multi-agent environment in place so Colab and Kaggle use the fixed PettingZoo-compatible implementation.


In [ ]:
from pathlib import Path

def patch_text_file(path: Path, replacements, must_remove=()):
    text = path.read_text(encoding="utf-8")
    if path.name == "multi_agent_env.py":
        already_patched = (
            'AgentSelector = getattr(_agent_selector, "AgentSelector", _agent_selector)' in text
            and 'self._agent_selector = agent_selector(ALL_AGENTS)' not in text
            and '_pending_rewards' not in text
        )
        if already_patched:
            return False
    changed = False
    for old, new in replacements:
        if old in text:
            text = text.replace(old, new)
            changed = True
    for marker in must_remove:
        if marker in text:
            raise RuntimeError(f"Patch for {path} did not remove marker: {marker}")
    if changed:
        path.write_text(text, encoding="utf-8")
    return changed

env_path = Path("env/multi_agent_env.py")
env_changed = patch_text_file(
    env_path,
    replacements=[
        (
            "from pettingzoo.utils import agent_selector",
            '''try:\n    # PettingZoo 1.25.0+ exposes the selector class as AgentSelector.\n    from pettingzoo.utils import AgentSelector\nexcept ImportError:\n    # Older releases expose agent_selector directly, while some transitional\n    # layouts expose a module with AgentSelector inside it.\n    from pettingzoo.utils import agent_selector as _agent_selector\n\n    AgentSelector = getattr(_agent_selector, "AgentSelector", _agent_selector)''',
        ),
        (
            "self._agent_selector = agent_selector(ALL_AGENTS)",
            "self._agent_selector = AgentSelector(ALL_AGENTS)",
        ),
        (
            '''        if self.terminations[agent] or self.truncations[agent]:\n            # Dead-step: PZ compliance requires we handle this\n            self._was_dead_step(action)\n            return\n''',
            '''        if self.terminations[agent] or self.truncations[agent]:\n            # Dead-step: PZ compliance requires we handle this\n            self._was_dead_step(action)\n            return\n        # The current agent's cumulative reward was already returned by last().\n        # Reset its accumulation window before processing a fresh action.\n        self._cumulative_rewards[agent] = 0.0\n        self._clear_rewards()\n''',
        ),
        (
            "        self._pending_rewards[RISK_MANAGER] = rm_reward",
            '''        # Defer emission until the Trader finishes the cycle so PettingZoo sees\n        # one reward publication per cycle.\n        self._rm_cycle_reward = float(rm_reward)''',
        ),
        (
            "        self._pending_rewards[PORTFOLIO_MGR] = 0.0  # Will be updated in _advance_market",
            "        # PM reward is deferred until after the trader executes and the outcome is known.",
        ),
        (
            "        self._pending_rewards[TRADER] = float(trader_reward)",
            "        self.rewards[TRADER] = float(trader_reward)",
        ),
        (
            "        self._pending_rewards[PORTFOLIO_MGR] = float(pm_reward)",
            "        self.rewards[PORTFOLIO_MGR] = float(pm_reward)",
        ),
        (
            "        self._pending_rewards[RISK_MANAGER] = float(self._pending_rewards.get(RISK_MANAGER, 0.0) + rm_pain)",
            "        self.rewards[RISK_MANAGER] = float(self._rm_cycle_reward + rm_pain)",
        ),
        (
            "            \"rewards\": dict(self._pending_rewards),",
            "            \"rewards\": dict(self.rewards),",
        ),
        (
            '''        self._prev_portfolio_value = new_value\n        self._pending_trade = None\n''',
            '''        self._prev_portfolio_value = new_value\n        self._pending_trade = None\n        self._rm_cycle_reward = 0.0\n''',
        ),
        (
            "        self._pending_rewards = {ag: 0.0 for ag in ALL_AGENTS}",
            "        self._rm_cycle_reward = 0.0",
        ),
        (
            '''    def _accumulate_rewards(self):\n        \"\"\"Move pending rewards into PZ cumulative reward tracking.\"\"\"\n        for ag in self.agents:\n            self.rewards[ag] = self._pending_rewards.get(ag, 0.0)\n            self._cumulative_rewards[ag] += self.rewards[ag]\n''',
            '''    def _accumulate_rewards(self):\n        \"\"\"Add the current step rewards into PettingZoo cumulative tracking.\"\"\"\n        for ag in self.agents:\n            self._cumulative_rewards[ag] += self.rewards[ag]\n''',
        ),
    ],
    must_remove=["self._agent_selector = agent_selector(ALL_AGENTS)", "_pending_rewards"],
)

train_path = Path("training/train_multi_agent.py")
train_changed = patch_text_file(
    train_path,
    replacements=[
        (
            '    print("  Multi-Agent Trading — Alternating Optimization Loop")',
            '    print("  Multi-Agent Trading - Alternating Optimization Loop")',
        ),
        (
            '    print("  Multi-Agent Trading \xe2\u20ac\u201d Alternating Optimization Loop")',
            '    print("  Multi-Agent Trading - Alternating Optimization Loop")',
        ),
        (
            '    print(f"  Episodes: {n_episodes}  |  Steps/ep: {max_steps_ep}  |  γ={gamma}")',
            '    print(f"  Episodes: {n_episodes}  |  Steps/ep: {max_steps_ep}  |  gamma={gamma}")',
        ),
        (
            '    print(f"  Episodes: {n_episodes}  |  Steps/ep: {max_steps_ep}  |  \xce\xb3={gamma}")',
            '    print(f"  Episodes: {n_episodes}  |  Steps/ep: {max_steps_ep}  |  gamma={gamma}")',
        ),
        (
            '            print(f"  → Checkpoint saved at episode {ep+1}")',
            '            print(f"  -> Checkpoint saved at episode {ep+1}")',
        ),
        (
            '            print(f"  \xe2\u2020\u2019 Checkpoint saved at episode {ep+1}")',
            '            print(f"  -> Checkpoint saved at episode {ep+1}")',
        ),
    ],
)

if env_changed:
    print(f"Patched {env_path} for hosted runtimes.")
else:
    print(f"{env_path} already contains the hosted-runtime fixes.")

if train_changed:
    print(f"Patched {train_path} for ASCII-safe console output.")
else:
    print(f"{train_path} already contains ASCII-safe console output.")


## 3. Validate The PettingZoo Multi-Agent Environment

Probe the constructor first so the notebook does not assume the wrong signature, then instantiate and inspect observation and action shapes.


In [ ]:
import inspect
import numpy as np

from env.multi_agent_env import (
    ALL_AGENTS,
    BASE_OBS_SIZE,
    MultiAgentTradingEnv,
    PM_MSG_SIZE,
    PORTFOLIO_MGR,
    RISK_MANAGER,
    RM_MSG_SIZE,
    TRADER,
)

print("MultiAgentTradingEnv signature:")
print(inspect.signature(MultiAgentTradingEnv))

env = MultiAgentTradingEnv(difficulty="easy", max_steps=50)
reset_result = env.reset(seed=7)

print("\nEnvironment summary")
print("-" * 60)
print(f"reset() returned: {reset_result}")
print(f"Agents: {env.agents}")
print(f"Turn order: RM -> PM -> Trader")
print(f"RM obs: {env.observe(RISK_MANAGER).shape} (base={BASE_OBS_SIZE})")
print(f"PM obs: {env.observe(PORTFOLIO_MGR).shape} (base+rm={BASE_OBS_SIZE + RM_MSG_SIZE})")
print(f"Trader obs: {env.observe(TRADER).shape} (base+rm+pm={BASE_OBS_SIZE + RM_MSG_SIZE + PM_MSG_SIZE})")
print(f"RM action space: {env.action_space(RISK_MANAGER)}")
print(f"PM action space: {env.action_space(PORTFOLIO_MGR)}")
print(f"Trader action space: {env.action_space(TRADER)}")


In [ ]:
env = MultiAgentTradingEnv(difficulty="easy", max_steps=10)
env.reset(seed=7)

rm_action = np.array([0.20, 1.0, 0.0], dtype=np.float32)
env.step(rm_action)

pm_action = np.array([0.35, 0.0], dtype=np.float32)
env.step(pm_action)

trader_action = {
    "direction": 1,
    "size": np.array([0.10], dtype=np.float32),
    "sl": np.array([0.0], dtype=np.float32),
    "tp": np.array([0.0], dtype=np.float32),
}
env.step(trader_action)

print("Completed one AEC cycle.")
print(f"Current step: {env._current_step}")
print(f"Current agent selection: {env.agent_selection}")
print(f"Latest rewards: {env.rewards}")


In [ ]:
import warnings

from pettingzoo.test import api_test

api_env = MultiAgentTradingEnv(difficulty="easy", max_steps=20)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="pettingzoo.test.api_test")
    api_test(api_env, num_cycles=20, verbose_progress=True)
print("PettingZoo API test passed.")


## 4. Run The Rule-Based Multi-Agent Trainer

This section exercises the multi-agent training loop without needing the GRPO stack.


In [ ]:
from training.train_multi_agent import train

metrics = train(
    n_episodes=40,
    max_steps_ep=150,
    gamma=0.99,
    alternating_freq=10,
    difficulty="easy",
    output_dir="outputs/multi_agent",
    save_every=20,
)

print(f"Saved training outputs to: {Path('outputs/multi_agent').resolve()}")
if metrics.get("trader_return"):
    print(f"Final trader return: {metrics['trader_return'][-1]:+.4f}")
if metrics.get("grade"):
    print(f"Final grade: {metrics['grade'][-1]:.4f}")


In [ ]:
from training.train_multi_agent import (
    RulePortfolioManagerPolicy,
    RuleRiskManagerPolicy,
    RuleTraderPolicy,
    collect_rollout,
)

policies = {
    RISK_MANAGER: RuleRiskManagerPolicy(),
    PORTFOLIO_MGR: RulePortfolioManagerPolicy(),
    TRADER: RuleTraderPolicy(),
}

print(f"{'Difficulty':<12} {'Episodes':<10} {'Mean Trader Return':<22} {'Mean PnL':<12} {'Mean DD':<12}")
print("-" * 74)

for diff in ["easy", "medium", "hard"]:
    returns, pnls, dds = [], [], []
    test_env = MultiAgentTradingEnv(difficulty=diff, max_steps=100)
    for _ in range(10):
        buffers, info = collect_rollout(test_env, policies, max_steps=100)
        returns.append(float(np.mean(buffers[TRADER].discounted_returns())))
        pnls.append(float(info.get("pnl_pct", 0.0)))
        dds.append(float(info.get("max_drawdown", 0.0)))
    print(f"{diff:<12} {10:<10} {np.mean(returns):+.6f}            {np.mean(pnls):+.4%}     {np.mean(dds):.4%}")


## 5. Generate Training Plots

Use the saved metrics when available, or fall back to the in-memory `metrics` object from the training cell.


In [ ]:
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

metrics_path = Path("outputs/multi_agent/metrics_final.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)

if not metrics_path.exists():
    if "metrics" not in globals():
        raise RuntimeError("Run the training cell first so metrics are available.")
    with open(metrics_path, "w", encoding="utf-8") as handle:
        json.dump(dict(metrics), handle, indent=2)

with open(metrics_path, encoding="utf-8") as handle:
    m = json.load(handle)

plots_dir = Path("plots")
plots_dir.mkdir(parents=True, exist_ok=True)
episodes = m["episode"]
n_eps = len(episodes)
print(f"Loaded {n_eps} episodes from {metrics_path}")

def smooth(values, window=10):
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid").tolist()

window = max(1, n_eps // 15)

fig, ax = plt.subplots(figsize=(12, 6))
trader_s = smooth(m["trader_return"], window)
rm_s = smooth(m["rm_return"], window)
pm_s = smooth(m["pm_return"], window)
ep_s = episodes[: len(trader_s)]
ax.plot(ep_s, trader_s, label="Trader", color="#2ecc71", linewidth=2)
ax.plot(ep_s, rm_s, label="Risk Manager", color="#e74c3c", linewidth=2)
ax.plot(ep_s, pm_s, label="Portfolio Manager", color="#3498db", linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Discounted Return")
ax.set_title("QuantHive: Per-Agent Reward Curves")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(plots_dir / "reward_curve.png", dpi=150)
plt.close(fig)

fig2, ax2 = plt.subplots(figsize=(12, 6))
pnl_s = smooth(m["pnl_pct"], window)
pnl_arr = np.array(pnl_s)
ax2.plot(episodes[: len(pnl_s)], pnl_s, color="#e74c3c", linewidth=2)
ax2.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax2.fill_between(episodes[: len(pnl_s)], 0, pnl_s, where=pnl_arr > 0, color="#2ecc71", alpha=0.2)
ax2.fill_between(episodes[: len(pnl_s)], 0, pnl_s, where=pnl_arr <= 0, color="#e74c3c", alpha=0.2)
ax2.set_xlabel("Episode")
ax2.set_ylabel("PnL %")
ax2.set_title("QuantHive: PnL Over Training")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
fig2.savefig(plots_dir / "loss_curve.png", dpi=150)
plt.close(fig2)

if n_eps >= 20:
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    names = ["Trader Return", "Grade", "Max Drawdown", "Sharpe"]
    early = [np.mean(m[key][:20]) for key in ["trader_return", "grade", "max_drawdown", "sharpe"]]
    late = [np.mean(m[key][-20:]) for key in ["trader_return", "grade", "max_drawdown", "sharpe"]]
    x = np.arange(len(names))
    ax3.bar(x - 0.175, early, 0.35, label="First 20 eps", color="#e74c3c", alpha=0.8)
    ax3.bar(x + 0.175, late, 0.35, label="Last 20 eps", color="#2ecc71", alpha=0.8)
    ax3.set_ylabel("Value")
    ax3.set_title("QuantHive: Early vs Late Training")
    ax3.set_xticks(x)
    ax3.set_xticklabels(names)
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    fig3.savefig(plots_dir / "baseline_comparison.png", dpi=150)
    plt.close(fig3)

print(f"Saved plots to: {plots_dir.resolve()}")


## 6. Preview Governance-Aware GRPO Prompts

Import only the lightweight prompt helpers so prompt preview does not depend on the trainer stack.


In [ ]:
from training.prompt_utils import build_prompt_multiagent, generate_pz_scenarios

scenarios = generate_pz_scenarios(n=3, difficulty="easy", max_env_steps=30)
print(f"Generated {len(scenarios)} scenarios.\n")

for i, scenario in enumerate(scenarios, start=1):
    print("=" * 60)
    print(f"Scenario {i}")
    print(
        f"RM: size_limit={scenario['rm_size_limit']:.2f}, "
        f"allow_new={scenario['rm_allow_new']}, force_reduce={scenario['rm_force_reduce']}"
    )
    print(
        f"PM: cap_alloc={scenario['pm_cap_alloc']:.2f}, "
        f"override={scenario['pm_override']:.2f}"
    )

print("\nFull prompt for scenario 1")
print("=" * 60)
print(build_prompt_multiagent(scenarios[0]))


In [ ]:
from env.reward import alignment_reward_func, format_reward_func, profit_reward_func
from training.grpo_verifiers_multiagent import (
    governance_reward_func_multiagent,
    risk_reward_func_multiagent,
)

test_prompt = build_prompt_multiagent(scenarios[0])
effective_limit = min(scenarios[0]["rm_size_limit"], scenarios[0]["pm_cap_alloc"])

compliant = (
    "<thought>\n"
    "Risk limits are active, so I will stay inside both the RM size cap and the PM allocation.\n"
    "</thought>\n"
    "<action>\n"
    '{{"direction": 1, "size": %.2f, "sl": 49000, "tp": 52000}}\n'
    "</action>"
) % (effective_limit * 0.7)

reckless = (
    "<thought>\nMarket is moving up. Going all in.\n</thought>\n"
    "<action>\n{\"direction\": 1, \"size\": 0.95, \"sl\": 0, \"tp\": 0}\n</action>"
)

prompts = [test_prompt, test_prompt]
completions = [compliant, reckless]

print(f"{'Verifier':<25} {'Compliant':<12} {'Reckless':<12}")
print("-" * 49)
for name, func in [
    ("Format", format_reward_func),
    ("Alignment", alignment_reward_func),
    ("Risk", risk_reward_func_multiagent),
    ("Profit", profit_reward_func),
    ("Governance", governance_reward_func_multiagent),
]:
    scores = func(prompts, completions)
    print(f"{name:<25} {scores[0]:<12.2f} {scores[1]:<12.2f}")


## 7. Optional GPU GRPO Training

Leave `RUN_GRPO` as `False` unless the runtime has a GPU and you want to install the heavier TRL and Unsloth stack.


In [ ]:
RUN_GRPO = False

if not RUN_GRPO:
    print("Set RUN_GRPO = True after enabling a GPU runtime.")
else:
    import json
    import torch
    from types import SimpleNamespace

    assert torch.cuda.is_available(), "GPU required for GRPO training"
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    extra_packages = [
        "datasets",
        "transformers",
        "trl",
        "peft",
        "accelerate",
        "safetensors",
        "unsloth",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *extra_packages])

    from datasets import Dataset
    from training.train_grpo_multiagent import load_model, make_trainer, save_model

    print("Generating 256 scenarios from MultiAgentTradingEnv...")
    scenarios = generate_pz_scenarios(n=256, difficulty="easy", max_env_steps=50)
    prompts = [{"prompt": build_prompt_multiagent(sc)} for sc in scenarios]
    dataset = Dataset.from_list(prompts)
    print(f"Dataset: {len(dataset)} prompts")

    model, tokenizer = load_model(
        "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
        max_seq_length=1024,
    )

    args = SimpleNamespace(
        output_dir="models/local_policy_grpo_multiagent",
        learning_rate=5e-5,
        per_device_batch_size=2,
        gradient_accumulation_steps=2,
        max_steps=100,
        save_steps=25,
        logging_steps=1,
        max_prompt_length=768,
        max_completion_length=200,
        num_generations=4,
    )

    trainer = make_trainer(model, tokenizer, dataset, args, torch)
    print(f"Starting GRPO training ({args.max_steps} steps)...")
    trainer.train()

    history = trainer.state.log_history
    rewards = [item["reward"] for item in history if "reward" in item]
    losses = [item["loss"] for item in history if "loss" in item]
    if not losses:
        losses = [0.0]

    from utils.plotting import plot_training_results

    plot_training_results(rewards, losses, output_dir="plots")
    save_model(model, tokenizer, args.output_dir)

    Path("outputs").mkdir(parents=True, exist_ok=True)
    with open("outputs/grpo_metrics.json", "w", encoding="utf-8") as handle:
        json.dump({"rewards": rewards, "losses": losses}, handle, indent=2)

    print(f"Model saved to {args.output_dir}")


## 8. Display Generated Artifacts


In [ ]:
try:
    from IPython.display import Image, Markdown, display
    has_ipython_display = True
except ImportError:
    has_ipython_display = False

plot_files = [
    ("plots/reward_curve.png", "Per-Agent Reward Curves"),
    ("plots/loss_curve.png", "PnL Or Policy Loss Curve"),
    ("plots/baseline_comparison.png", "Early vs Late Training"),
]

for path, title in plot_files:
    if Path(path).exists():
        if has_ipython_display:
            display(Markdown(f"### {title}"))
            display(Image(filename=path, width=700))
        else:
            print(f"{title}: {Path(path).resolve()}")
    else:
        print(f"Missing: {path}")


## 9. Submission Checklist


In [ ]:
import yaml

checks = [
    ("openenv.yaml", Path("openenv.yaml")),
    ("README.md", Path("README.md")),
    ("WRITEUP.md", Path("WRITEUP.md")),
    ("Multi-agent env", Path("env/multi_agent_env.py")),
    ("REINFORCE trainer", Path("training/train_multi_agent.py")),
    ("GRPO trainer", Path("training/train_grpo_multiagent.py")),
    ("Prompt helpers", Path("training/prompt_utils.py")),
    ("GRPO verifiers", Path("training/grpo_verifiers_multiagent.py")),
    ("Training notebook", Path("mate_training.ipynb")),
    ("Reward curve", Path("plots/reward_curve.png")),
    ("Loss curve", Path("plots/loss_curve.png")),
    ("Baseline comparison", Path("plots/baseline_comparison.png")),
    ("Dockerfile", Path("Dockerfile")),
    ("requirements-space.txt", Path("requirements-space.txt")),
]

print(f"{'Deliverable':<28} {'Status':<10}")
print("-" * 42)
for name, path in checks:
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:<28} {status:<10}")

with open("openenv.yaml", encoding="utf-8") as handle:
    manifest = yaml.safe_load(handle)
entry_point = manifest.get("environment", {}).get("entry_point", "")
print(f"\nopenenv.yaml entry_point: {entry_point}")
print(f"PettingZoo env configured: {'multi_agent_env' in entry_point}")
